In [4]:
import os
import json
import time
from pathlib import Path
from typing import Dict, Any, Tuple, Optional

import requests
import pandas as pd

In [5]:
TOKEN = os.getenv("ENSEMBLEDATA_TOKEN", "dD5Nd9L5rPcu743S")
ROOT = "https://ensembledata.com/apis"

In [13]:
# -----------------------
# API calls (SCRAPE ONLY)
# -----------------------
def get_user_info(username: str) -> Dict[str, Any]:
    endpoint = "/instagram/user/info"
    params = {"username": username, "token": TOKEN}
    r = requests.get(ROOT + endpoint, params=params, timeout=60)
    r.raise_for_status()
    return r.json()

def get_user_detailed_info(username: str) -> Dict[str, Any]:
    endpoint = "/instagram/user/detailed-info"
    params = {"username": username, "token": TOKEN}
    r = requests.get(ROOT + endpoint, params=params, timeout=60)
    r.raise_for_status()
    return r.json()

In [14]:
# -----------------------
# Save / load raw payloads (JSON per account)
# -----------------------
def save_json(payload: Dict[str, Any], out_path: Path) -> None:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with out_path.open("w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)

def load_json(path: Path) -> Dict[str, Any]:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)

def safe_username(name: str) -> str:
    # keep filenames clean
    return "".join(ch for ch in name.strip() if ch.isalnum() or ch in ("_", "-", "."))

In [34]:

# -----------------------
# Parsers (PARSE ONLY)
# -----------------------
def parse_user_info_payload(payload: Dict[str, Any]) -> pd.DataFrame:
    d = payload.get("data", {}) or {}
    row = {
        "username": d.get("username", ""),
        "pk": d.get("pk", ""),
        "pk_id": d.get("pk_id", ""),
        "full_name": d.get("full_name", ""),
        "is_private": d.get("is_private", None),
        "is_verified": d.get("is_verified", None),
        "profile_pic_url": d.get("profile_pic_url", ""),
        "profile_pic_id": d.get("profile_pic_id", None),
        "has_highlight_reels": d.get("has_highlight_reels", None),
        "has_opt_eligible_shop": d.get("has_opt_eligible_shop", None),
        "latest_reel_media": d.get("latest_reel_media", None),
        "live_broadcast_id": d.get("live_broadcast_id", None),
        "should_show_category": d.get("should_show_category", None),
    }
    return pd.DataFrame([row])

def parse_user_detailed_payload(payload: Dict[str, Any]) -> pd.DataFrame:
    d = payload.get("data", {}) or {}

    followers = (d.get("edge_followed_by") or {}).get("count", None)
    following = (d.get("edge_follow") or {}).get("count", None)
    posts_count = (d.get("edge_owner_to_timeline_media") or {}).get("count", None)

    bio_links = d.get("bio_links") or []
    bio_links_count = len(bio_links) if isinstance(bio_links, list) else None

    row = {
        "username": d.get("username", ""),
        "user_id": d.get("id", ""),
        "fbid": d.get("fbid", None),
        "fb_profile_biolink": d.get("fb_profile_biolink", None),
        "category_name": d.get("category_name", None),
        "is_verified": d.get("is_verified", None),
        "is_business_account": d.get("is_business_account", None),
        "is_professional_account": d.get("is_professional_account", None),
        "biography": d.get("biography", None),
        "external_url": d.get("external_url", None),
        "followers": followers,
        "following": following,
        "posts_count": posts_count,
        "bio_links_count": bio_links_count,
        "profile_pic_url_hd": d.get("profile_pic_url_hd", None),
        "is_private_detailed": d.get("is_private", None),
        #"is_verified_detailed": d.get("is_verified", None),
        "business_phone_number": d.get("business_phone_number", None),
        "business_email": d.get("business_email", None),
        "business_address_json": d.get("business_address_json", None)
    }
    return pd.DataFrame([row])

In [16]:

# -----------------------
# CSV input
# -----------------------
def read_accounts_csv(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    needed = {"brand", "username"}
    missing = needed - set(df.columns.str.lower())
    # handle case where user wrote Brands/Username etc.
    if missing:
        # normalize column names
        df.columns = [c.strip().lower() for c in df.columns]
        missing2 = needed - set(df.columns)
        if missing2:
            raise ValueError(f"CSV must contain columns: {sorted(list(needed))}. Missing: {sorted(list(missing2))}")

    # normalize and drop blanks
    df["brand"] = df["brand"].astype(str).str.strip()
    df["username"] = df["username"].astype(str).str.strip()
    df = df[(df["username"] != "") & (df["username"].str.lower() != "nan")].copy()
    return df

In [17]:

# -----------------------
# Step 1: SCRAPE + SAVE RAW JSONS
# -----------------------
def scrape_and_save_raw(
    accounts_csv: str,
    out_root: str = "outputs",
    sleep_s: float = 0.3,
) -> Tuple[Path, Path]:
    out_root = Path(out_root)
    user_info_dir = out_root / "user_info"
    user_detailed_dir = out_root / "user_info_detailed"

    df_accounts = read_accounts_csv(accounts_csv)

    for _, row in df_accounts.iterrows():
        username = row["username"]
        uname = safe_username(username)

        # --- user/info ---
        try:
            info_payload = get_user_info(username)
            save_json(info_payload, user_info_dir / f"{uname}.json")
        except Exception as e:
            # save an error json so you can audit failures later
            save_json({"username": username, "error": str(e)}, user_info_dir / f"{uname}.error.json")

        time.sleep(sleep_s)

        # --- user/detailed-info ---
        try:
            detailed_payload = get_user_detailed_info(username)
            save_json(detailed_payload, user_detailed_dir / f"{uname}.json")
        except Exception as e:
            save_json({"username": username, "error": str(e)}, user_detailed_dir / f"{uname}.error.json")

        time.sleep(sleep_s)

    return user_info_dir, user_detailed_dir

In [18]:
if __name__ == "__main__":
    accounts_csv = "accounts.csv"   # CSV with columns: brands, username
    out_root = "outputs"
    sleep_s = 0.3

    scrape_and_save_raw(
        accounts_csv=accounts_csv,
        out_root=out_root,
        sleep_s=sleep_s
    )

    print(f"[DONE] Scraping finished. Raw JSON saved under: {out_root}/user_info and {out_root}/user_info_detailed")

[DONE] Scraping finished. Raw JSON saved under: outputs/user_info and outputs/user_info_detailed


In [35]:

# -----------------------
# Step 2: PARSE saved JSONS -> tables (+ merged)
# -----------------------
def parse_saved_jsons_to_tables(
    accounts_csv: str,
    in_root: str = "outputs",
    out_root: str = "outputs",
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    in_root = Path(in_root)
    out_root = Path(out_root)
    user_info_dir = in_root / "user_info"
    user_detailed_dir = in_root / "user_info_detailed"

    df_accounts = read_accounts_csv(accounts_csv)

    info_rows = []
    detailed_rows = []

    for _, row in df_accounts.iterrows():
        username = row["username"]
        brand = row["brand"]
        uname = safe_username(username)

        # load raw jsons (skip if missing)
        info_path = user_info_dir / f"{uname}.json"
        detailed_path = user_detailed_dir / f"{uname}.json"

        if info_path.exists():
            payload = load_json(info_path)
            df1 = parse_user_info_payload(payload)
            df1.insert(0, "brand", brand)
            info_rows.append(df1)
        else:
            # placeholder row (optional)
            info_rows.append(pd.DataFrame([{"brand": brand, "username": username}]))

        if detailed_path.exists():
            payload = load_json(detailed_path)
            df2 = parse_user_detailed_payload(payload)
            df2.insert(0, "brand", brand)
            detailed_rows.append(df2)
        else:
            detailed_rows.append(pd.DataFrame([{"brand": brand, "username": username}]))

    user_info_df = pd.concat(info_rows, ignore_index=True) if info_rows else pd.DataFrame()
    user_detailed_df = pd.concat(detailed_rows, ignore_index=True) if detailed_rows else pd.DataFrame()

    merged_df = user_info_df.merge(
        user_detailed_df,
        on=["brand", "username"],
        how="left",
        suffixes=("_info", "_detailed"),
    )

    # Save parsed tables
    out_root.mkdir(parents=True, exist_ok=True)
    user_info_df.to_csv(out_root / "user_info_table.csv", index=False)
    user_detailed_df.to_csv(out_root / "user_detailed_table.csv", index=False)
    merged_df.to_csv(out_root / "user_merged_table.csv", index=False)

    return user_info_df, user_detailed_df, merged_df

In [36]:

# -----------------------
# Example CLI usage
# -----------------------
if __name__ == "__main__":
    # INPUT CSV with columns: brands, username
    accounts_csv = "accounts.csv"

    # Step 2: parse saved jsons into tables + merged
    user_info_df, user_detailed_df, merged_df = parse_saved_jsons_to_tables(
        accounts_csv=accounts_csv,
        in_root="outputs",
        out_root="outputs",
    )

    print("\nUSER_INFO (parsed)")
    #print(user_info_df.head(10).to_string(index=False))

    print("\nUSER_DETAILED_INFO (parsed)")
    #print(user_detailed_df.head(10).to_string(index=False))

    print("\nMERGED (parsed)")
    #print(merged_df.head(10).to_string(index=False))
    


USER_INFO (parsed)

USER_DETAILED_INFO (parsed)

MERGED (parsed)


In [37]:
user_detailed_df

,brand,username,user_id,fbid,fb_profile_biolink,category_name,is_verified,is_business_account,is_professional_account,biography,external_url,followers,following,posts_count,bio_links_count,profile_pic_url_hd,is_private_detailed,business_phone_number,business_email,business_address_json
0,adidas,adidas,20269764,17841401372920316,None,Product/service,True,True,True,#YouGotThis,https://linktr.ee/adidas,29470913,1013,1381,1,https://instagram.fbru2-1.fna.fbcdn.net/v/t51....,False,None,None,"{""city_name"": ""Herzogenaurach"", ""city_id"": 103..."
